# Querying and Filtering

## Overview
MongoDB's query language uses Python dictionaries (when using PyMongo) where keys are field names and values are either exact match values or operator documents starting with `$`.

**Query structure:**
```python
collection.find(
    filter,      # Which documents to return (like SQL WHERE)
    projection   # Which fields to include/exclude (like SQL SELECT column list)
)
```

**Basic filter:** `{"province": "NB"}` — exact match on a field  
**Operator filter:** `{"age": {"$gt": 40}}` — field with comparison operator  
**Multiple conditions:** `{"province": "NB", "sex": "F"}` — implicit AND (like SQL WHERE a AND b)

**SQL to MongoDB filter translation:**

| SQL | MongoDB |
|---|---|
| `WHERE province = 'NB'` | `{"province": "NB"}` |
| `WHERE age > 40` | `{"age": {"$gt": 40}}` |
| `WHERE province IN ('NB', 'ON')` | `{"province": {"$in": ["NB", "ON"]}}` |
| `WHERE col IS NOT NULL` | `{"col": {"$exists": True}}` |
| `WHERE a = 1 AND b = 2` | `{"a": 1, "b": 2}` |
| `WHERE a = 1 OR b = 2` | `{"$or": [{"a": 1}, {"b": 2}]}` |
| `WHERE name LIKE 'A%'` | `{"name": {"$regex": "^A"}}` |

---

In [1]:

print("=== Comparison and logical operators ===")
ops = [
    ("$eq",   "Equal",                   "{'sex': {'$eq': 'F'}}",              "same as {'sex': 'F'}"),
    ("$ne",   "Not equal",               "{'sex': {'$ne': 'M'}}",              ""),
    ("$gt",   "Greater than",            "{'age': {'$gt': 40}}",               ""),
    ("$gte",  "Greater than or equal",   "{'age': {'$gte': 18}}",              ""),
    ("$lt",   "Less than",               "{'score': {'$lt': 100}}",            ""),
    ("$lte",  "Less than or equal",      "{'score': {'$lte': 50}}",            ""),
    ("$in",   "Value in list",           "{'province': {'$in': ['NB','ON']}}", "SQL: IN (...)"),
    ("$nin",  "Value not in list",       "{'status': {'$nin': ['cancelled']}}", "SQL: NOT IN"),
    ("$and",  "All conditions true",     "{'$and': [{...},{...}]}",            "default when multiple keys"),
    ("$or",   "Any condition true",      "{'$or': [{'sex':'F'},{'age':{'$lt':30}}]}", "SQL: OR"),
    ("$not",  "Negate operator",         "{'age': {'$not': {'$gt': 65}}}",     ""),
    ("$nor",  "None of the conditions",  "{'$nor': [{...},{...}]}",            "SQL: NOT (x OR y)"),
    ("$exists","Field exists",           "{'phone': {'$exists': True}}",       "SQL: col IS NOT NULL"),
    ("$type", "Field BSON type",         "{'dob': {'$type': 'string'}}",       ""),
]
print(f"  {'Operator':<8s}  {'Meaning':<26s}  {'Example':<42s}  Note")
print("  " + "-"*90)
for op, meaning, example, note in ops:
    print(f"  {op:<8s}  {meaning:<26s}  {example:<42s}  {note}")


=== Comparison and logical operators ===
  Operator  Meaning                     Example                                     Note
  ------------------------------------------------------------------------------------------
  $eq       Equal                       {'sex': {'$eq': 'F'}}                       same as {'sex': 'F'}
  $ne       Not equal                   {'sex': {'$ne': 'M'}}                       
  $gt       Greater than                {'age': {'$gt': 40}}                        
  $gte      Greater than or equal       {'age': {'$gte': 18}}                       
  $lt       Less than                   {'score': {'$lt': 100}}                     
  $lte      Less than or equal          {'score': {'$lte': 50}}                     
  $in       Value in list               {'province': {'$in': ['NB','ON']}}          SQL: IN (...)
  $nin      Value not in list           {'status': {'$nin': ['cancelled']}}         SQL: NOT IN
  $and      All conditions true         {'$and': [{..

---
## Array and nested field operators

In [2]:

print("=== Array and nested field operators ===")
array_ops = '''
# ── Query on array field (conditions) ───────────────────────────
# Documents where conditions array contains 'hypertension':
patients.find({"conditions": "hypertension"})

# Documents where conditions contains BOTH values ($all):
patients.find({"conditions": {"$all": ["hypertension", "type2_diabetes"]}})

# Documents where conditions has exactly 2 elements:
patients.find({"conditions": {"$size": 2}})

# Documents where ANY element in conditions matches:
patients.find({"conditions": {"$elemMatch": {"$regex": "^hyper"}}})

# ── Nested field (dot notation) ──────────────────────────────────
# contact is a subdocument: {"email": "...", "phone": "..."}
patients.find({"contact.email": {"$exists": True}})
patients.find({"contact.phone": {"$regex": "^506"}})   # NB area code

# ── Array of subdocuments ────────────────────────────────────────
# If encounters is an array of objects like:
# [{"date": "2023-04-10", "type": "outpatient", "provider_id": "D002"}, ...]
db.patients.find({"encounters.type": "outpatient"})
db.patients.find({"encounters": {"$elemMatch": {
    "type": "outpatient",
    "date": {"$gte": "2023-01-01"}
}}})
'''
print(array_ops)


=== Array and nested field operators ===

# ── Query on array field (conditions) ───────────────────────────
# Documents where conditions array contains 'hypertension':
patients.find({"conditions": "hypertension"})

# Documents where conditions contains BOTH values ($all):
patients.find({"conditions": {"$all": ["hypertension", "type2_diabetes"]}})

# Documents where conditions has exactly 2 elements:
patients.find({"conditions": {"$size": 2}})

# Documents where ANY element in conditions matches:
patients.find({"conditions": {"$elemMatch": {"$regex": "^hyper"}}})

# ── Nested field (dot notation) ──────────────────────────────────
# contact is a subdocument: {"email": "...", "phone": "..."}
patients.find({"contact.email": {"$exists": True}})
patients.find({"contact.phone": {"$regex": "^506"}})   # NB area code

# ── Array of subdocuments ────────────────────────────────────────
# If encounters is an array of objects like:
# [{"date": "2023-04-10", "type": "outpatient", "provider_id": "

---
## Sorting, limiting, updating, and deleting

In [3]:

print("=== Sorting, limiting, skipping, and updating ===")
query_controls = '''
# ── Sort ─────────────────────────────────────────────────────────
patients.find().sort("last_name", 1)          # 1 = ascending (ASCENDING)
patients.find().sort("dob", -1)               # -1 = descending (DESCENDING)
patients.find().sort([("province", 1), ("last_name", 1)])  # multi-key sort

# ── Limit and skip ───────────────────────────────────────────────
patients.find().limit(10)                     # first 10 documents
patients.find().skip(20).limit(10)            # documents 21-30 (pagination)

# ── Combined ─────────────────────────────────────────────────────
results = (
    patients
    .find({"province": "NB"}, {"first_name": 1, "last_name": 1, "_id": 0})
    .sort("last_name", 1)
    .limit(5)
)

# ── Update one ───────────────────────────────────────────────────
patients.update_one(
    {"patient_id": "P001"},                   # filter
    {"$set": {"province": "NS"},              # $set: update only these fields
     "$push": {"conditions": "pre_diabetes"}} # $push: append to array
)

# ── Update many ──────────────────────────────────────────────────
patients.update_many(
    {"province": "NB"},
    {"$set": {"region": "Atlantic"}}          # add a new field to all NB patients
)

# ── Upsert: insert if not exists, update if exists ───────────────
patients.update_one(
    {"patient_id": "P999"},
    {"$setOnInsert": {"first_name": "New", "last_name": "Patient", "province": "ON"}},
    upsert=True
)

# ── Delete ───────────────────────────────────────────────────────
patients.delete_one({"patient_id": "P999"})
patients.delete_many({"province": "BC", "active": False})
'''
print(query_controls)

print("Common update operators:")
update_ops = [
    ("$set",        "Set field values (add or update)"),
    ("$unset",      "Remove a field"),
    ("$inc",        "Increment a numeric field: {'$inc': {'visit_count': 1}}"),
    ("$push",       "Append to an array"),
    ("$pull",       "Remove matching elements from an array"),
    ("$addToSet",   "Append to array only if value not already present"),
    ("$rename",     "Rename a field"),
    ("$currentDate","Set a field to the current date/timestamp"),
]
for op, desc in update_ops:
    print(f"  {op:<16s}  {desc}")


=== Sorting, limiting, skipping, and updating ===

# ── Sort ─────────────────────────────────────────────────────────
patients.find().sort("last_name", 1)          # 1 = ascending (ASCENDING)
patients.find().sort("dob", -1)               # -1 = descending (DESCENDING)
patients.find().sort([("province", 1), ("last_name", 1)])  # multi-key sort

# ── Limit and skip ───────────────────────────────────────────────
patients.find().limit(10)                     # first 10 documents
patients.find().skip(20).limit(10)            # documents 21-30 (pagination)

# ── Combined ─────────────────────────────────────────────────────
results = (
    patients
    .find({"province": "NB"}, {"first_name": 1, "last_name": 1, "_id": 0})
    .sort("last_name", 1)
    .limit(5)
)

# ── Update one ───────────────────────────────────────────────────
patients.update_one(
    {"patient_id": "P001"},                   # filter
    {"$set": {"province": "NS"},              # $set: update only these fields
     "

---
## Common Pitfalls

**1. Querying an array field with $eq instead of direct value match**
`{'conditions': {'$eq': 'hypertension'}}` and `{'conditions': 'hypertension'}` are equivalent -- MongoDB automatically checks whether the array contains the value. But `{'conditions': ['hypertension']}` (with square brackets) means the array is exactly `['hypertension']` with no other elements. The distinction between element match and exact array match is a common source of missed results.

**2. Dot notation on arrays without $elemMatch**
`{'encounters.type': 'outpatient', 'encounters.date': {'$gte': '2023-01-01'}}` does NOT require both conditions to match the same array element. It matches any document where at least one encounter has `type: outpatient` AND at least one (possibly different) encounter has a date >= 2023-01-01. Use `$elemMatch` to require both conditions on the same element.

**3. Using skip() for deep pagination on large collections**
`.skip(100000).limit(10)` scans and discards 100,000 documents before returning 10. This is O(N) and gets slower with every page. Use range-based pagination instead: store the `_id` of the last document on the previous page and filter `{'_id': {'$gt': last_id}}`.

**4. $set on a nested field replacing the whole subdocument**
`{'$set': {'contact': {'email': 'new@email.com'}}}` replaces the entire `contact` subdocument, deleting the `phone` field. Use dot notation to update a single subfield: `{'$set': {'contact.email': 'new@email.com'}}`.

**5. Not using $regex anchors causing unexpectedly broad matches**
`{'last_name': {'$regex': 'son'}}` matches Anderson, Johnson, Johansson, and Wilson. Always anchor with `^` for prefix or `$` for suffix when you mean a specific pattern. Also add `'$options': 'i'` for case-insensitive matching.

**6. update_many without a filter updating the entire collection**
`patients.update_many({}, {'$set': {'migrated': True}})` updates every document in the collection. An empty filter `{}` in update_many, delete_many, or find matches all documents. Always double-check that your filter is intentionally broad before running a multi-document write.


---
*sql_methods_library - Samantha McGarrigle*